# Lesson 05: 물체 간 충돌 - 볼링 시뮬레이션

## 학습 목표
1. 초기 속도(`SetPosDt`)로 물체 발사하기
2. 여러 물체 간 상호 충돌
3. 물체를 프로그래밍으로 대량 배치
4. 충돌 후 결과 분석

## 새로 배우는 Chrono API
| 메서드 | 역할 |
|---|---|
| `SetPosDt(ChVector3d)` | 초기 속도 설정 (발사!) |
| `GetPosDt()` | 현재 속도 읽기 |

> `PosDt` = Position Derivative = 위치의 시간 미분 = 속도

In [ ]:
import pychrono as chrono

sys = chrono.ChSystemNSC()
sys.SetGravitationalAcceleration(chrono.ChVector3d(0, -9.81, 0))
sys.SetCollisionSystemType(chrono.ChCollisionSystem.Type_BULLET)

material = chrono.ChContactMaterialNSC()
material.SetFriction(0.4)
material.SetRestitution(0.3)

## 바닥 + 볼링 핀 배치

핀 10개를 피라미드 형태로 배치:
```
    O        ← 1열 (1개)
   O O       ← 2열 (2개)
  O O O      ← 3열 (3개)
 O O O O     ← 4열 (4개)
```

In [ ]:
# 바닥
floor = chrono.ChBody()
floor.SetPos(chrono.ChVector3d(0, -0.5, 0))
floor.SetFixed(True)
floor.EnableCollision(True)
floor.AddCollisionShape(chrono.ChCollisionShapeBox(material, 40, 1, 20))
sys.AddBody(floor)

# 볼링 핀 좌표 (x, z)
pin_positions = [
    (3, 0),                                    # 1열
    (3.5, -0.3), (3.5, 0.3),                   # 2열
    (4.0, -0.6), (4.0, 0), (4.0, 0.6),         # 3열
    (4.5, -0.9), (4.5, -0.3), (4.5, 0.3), (4.5, 0.9),  # 4열
]

pins = []
for px, pz in pin_positions:
    pin = chrono.ChBodyEasyCylinder(
        chrono.ChAxis_Y, 0.06, 0.4, 700, True, True, material
    )
    pin.SetPos(chrono.ChVector3d(px, 0.2, pz))
    sys.AddBody(pin)
    pins.append(pin)

print(f"볼링 핀 {len(pins)}개 배치 완료")

## 볼링공: 초기 속도로 발사

- `SetPosDt(ChVector3d(vx, vy, vz))` = 초기 속도 벡터 설정
- x방향으로 6 m/s → 핀을 향해 굴러감
- 운동량 = 질량 × 속도 → 무거울수록, 빠를수록 핀이 잘 넘어감

In [ ]:
ball = chrono.ChBodyEasySphere(0.15, 12000, True, True, material)
ball.SetPos(chrono.ChVector3d(-5, 0.15, 0))   # 왼쪽에서 시작

# ★ 초기 속도 설정: x방향으로 6 m/s
ball.SetPosDt(chrono.ChVector3d(6, 0, 0))

sys.AddBody(ball)

print(f"볼링공: 질량={ball.GetMass():.2f} kg")
print(f"초기 속도: 6.0 m/s (→)")
print(f"운동량: {ball.GetMass() * 6.0:.1f} kg·m/s")

## 시뮬레이션 실행 + 결과 분석

In [ ]:
# 5초간 시뮬레이션
while sys.GetChTime() < 5.0:
    sys.DoStepDynamics(0.005)

# 쓰러진 핀 세기: 원래 높이(y=0.2)에서 크게 벗어났으면 쓰러진 것
fallen = sum(1 for p in pins if abs(p.GetPos().y - 0.2) > 0.1)

print(f"시뮬레이션 {sys.GetChTime():.2f}초 완료")
print(f"쓰러진 핀: {fallen} / {len(pins)}")
print(f"볼링공 최종 위치: ({ball.GetPos().x:.1f}, {ball.GetPos().y:.1f}, {ball.GetPos().z:.1f})")
print(f"볼링공 최종 속도: {ball.GetPosDt().x:.2f} m/s (충돌 후 감속)")

## 실험해보기

1. **속도를 바꾸면?** `SetPosDt(ChVector3d(3, 0, 0))` → 느린 공, `(12, 0, 0)` → 빠른 공
2. **공 크기/질량을 바꾸면?** 밀도를 높이거나 반지름을 키우면?
3. **비스듬히 던지면?** `SetPosDt(ChVector3d(6, 0, 0.5))` → 약간 옆으로
4. **핀 간격을 바꾸면?** `pin_positions` 수정
5. **핀 재질을 바꾸면?** `SetRestitution(0.9)` → 핀이 더 많이 튕겨나감